# Text Statistics Analyzer

**Difficulty:** Intermediate | **Teacher:** Petra

Pythonic thinking is about elegance and readability!

Text analysis is everywhere — search engines, writing tools, spam filters, you name it. Today you'll build a function that crunches the numbers on a block of text and returns a neat summary.

Write a function `analyze_text(text)` that takes a string and returns a **dictionary** with the following statistics:

- `word_count` — total number of words
- `unique_words` — number of distinct words (case-insensitive)
- `sentence_count` — number of sentences (split on `.`, `!`, or `?`)
- `avg_word_length` — average word length, rounded to 2 decimal places
- `most_common_word` — the most frequently occurring word (case-insensitive, ignore punctuation)

Then write a second function `print_report(stats)` that prints the results in a readable format using f-strings.

## Requirements

- Strip punctuation from words before counting (e.g. `"hello,"` → `"hello"`)
- Word comparison is case-insensitive (`"Python"` and `"python"` are the same word)
- Empty strings or whitespace-only input should return sensible defaults (zeros, empty string for most_common_word)
- Sentences are defined by `.`, `!`, or `?` — ignore empty segments between them

## Example

```python
text = "Python is great. I love Python! Do you love Python?"
stats = analyze_text(text)
# {
#   'word_count': 10,
#   'unique_words': 7,
#   'sentence_count': 3,
#   'avg_word_length': 3.90,
#   'most_common_word': 'python'
# }
```

## Hints

- `str.translate()` + `str.maketrans()` can strip punctuation cleanly
- The `re` module's `re.split()` handles splitting on multiple delimiters
- Dictionaries are perfect for counting — or check out `collections.Counter`!
- `str.split()` handles multiple spaces gracefully

In [ ]:
import re
from collections import Counter

PATTERN = r'\W+'
SENTENCES = r'(?<=[.!?])\s+'

def analyze_text(text: str) -> dict:
    '''Return a statistics dictionary for the given text.'''

    # split text into words / sentences
    # ignore empty words
    words = [w for w in re.split(PATTERN, text) if w]
    sentences = [s for s in re.split(SENTENCES, text) if s]
    counts = Counter(words)

    if not text:
        return {
            'word_count': 0,
            'unique_words': 0,
            'sentence_count': 0,
            'avg_word_length': 0,
            'most_common_word': ''
        }
    
    word_count = counts.total()
    unique_words = len(list(set(words)))
    sentence_count = len(sentences)
    avg_word_lenth = sum([len(w) for w in words]) / word_count
    most_common_word, _ = counts.most_common(1)[0]

    stats = {
    'word_count': word_count,
    'unique_words': unique_words,
    'sentence_count': sentence_count,
    'avg_word_length': avg_word_lenth,
    'most_common_word': most_common_word.lower()
    }
    
    return stats

def print_report(stats: dict) -> None:
    '''Print a formatted report of the text statistics.'''
    for k, v in stats.items():
        print(f'{k}: {v}')


text1 = "Python is great. I love Python! Do you love Python?"
stats1 = analyze_text(text1)
print_report(stats1)

word_count: 10
unique_words: 7
sentence_count: 3
avg_word_length: 3.9
most_common_word: python


In [37]:
# Test cases — run these to check your solution

text1 = "Python is great. I love Python! Do you love Python?"
stats1 = analyze_text(text1)
assert stats1['word_count'] == 10, f"Expected 10, got {stats1['word_count']}"
assert stats1['unique_words'] == 7, f"Expected 7, got {stats1['unique_words']}"
assert stats1['sentence_count'] == 3, f"Expected 3, got {stats1['sentence_count']}"
assert stats1['most_common_word'] == 'python', f"Expected 'python', got {stats1['most_common_word']}"
print("Test 1 passed!")

text2 = ""
stats2 = analyze_text(text2)
assert stats2['word_count'] == 0
assert stats2['most_common_word'] == ''
print("Test 2 passed!")

text3 = "Hello, world! Hello, world! Hello!"
stats3 = analyze_text(text3)
assert stats3['most_common_word'] == 'hello'
assert stats3['sentence_count'] == 3
print("Test 3 passed!")

print("\n--- Report for text1 ---")
print_report(stats1)

Test 1 passed!
Test 2 passed!
Test 3 passed!

--- Report for text1 ---
word_count: 10
unique_words: 7
sentence_count: 3
avg_word_length: 3.9
most_common_word: python


---
# Feedback — Petra

Great work getting all three tests green! Your instincts were solid: `re.split(r'\W+', ...)` is exactly the right tool for stripping punctuation cleanly, and reaching for `Counter` right away is very Pythonic. Let's look at a few things to tighten it up.

## What worked well

- **`\W+` pattern for tokenizing** — clean, idiomatic, handles punctuation and whitespace in one shot
- **`Counter` + `most_common(1)`** — exactly what it's designed for, nice tuple unpacking too
- **Type hints on both functions** — good habit at intermediate level
- **Handling the empty string case** — you thought about edge cases!

## Areas for improvement

### 1. Case-insensitivity bug (hidden by the tests)

Your words are split from the raw text, so `"Python"` and `"python"` land in the `Counter` as *different* keys. The tests happen to use consistent casing, so this slips through — but try `"Python python PYTHON"` and you'd get `unique_words: 3` instead of `1`.

Fix: lowercase *before* counting.

```python
words = [w.lower() for w in re.split(PATTERN, text) if w]
```

Once words are already lowercase, the `.lower()` on `most_common_word` at the end becomes unnecessary too.

### 2. `avg_word_length` isn't rounded

The spec says "rounded to 2 decimal places" — you're missing the `round()` call. Currently `3.9` comes back instead of `3.90` (Python floats won't print trailing zeros, but `round(x, 2)` ensures the value is correctly rounded at 2 decimal places).

```python
avg_word_length = round(sum(len(w) for w in words) / word_count, 2)
```

### 3. Small Pythonic tweaks

- `len(list(set(words)))` → `len(set(words))` — `len()` works on sets directly, no need to convert
- `sum([len(w) for w in words])` → `sum(len(w) for w in words)` — drop the brackets to use a generator expression; more memory-efficient and more idiomatic
- `avg_word_lenth` is a typo (missing the `g`)!
- The early-return for empty text comes *after* the regex splits — safe, but slightly wasteful. A `if not text.strip(): return {...}` at the very top is cleaner.

## Alternative approach

Consider consolidating the logic now that words are pre-lowercased — everything flows from a single clean list:

In [ ]:
import re
from collections import Counter

PATTERN = r'\W+'
SENTENCES = r'(?<=[.!?])\s+'

def analyze_text(text: str) -> dict:
    if not text.strip():
        return {'word_count': 0, 'unique_words': 0, 'sentence_count': 0,
                'avg_word_length': 0.0, 'most_common_word': ''}

    words = [w.lower() for w in re.split(PATTERN, text) if w]
    sentences = [s for s in re.split(SENTENCES, text) if s]
    counts = Counter(words)

    return {
        'word_count': len(words),
        'unique_words': len(set(words)),
        'sentence_count': len(sentences),
        'avg_word_length': round(sum(len(w) for w in words) / len(words), 2),
        'most_common_word': counts.most_common(1)[0][0],
    }


def print_report(stats: dict) -> None:
    for k, v in stats.items():
        print(f'{k}: {v}')
